<div align='center'>

# Practica 5

<img src='https://media2.giphy.com/media/v1.Y2lkPTc5MGI3NjExaHhlMTBkeWt4NGtjcTJ3Znc1MTlueDI0dm1mOG1tZ2t6ODNmYmw5aSZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/uHox9Jm5TyTPa/giphy.gif'>

</div>

## Creamos el entorno virtual
Vamos a tener que instalar el jdk y configurar las variables de entorno


In [ ]:
from pyspark.sql import SparkSession
import os, sys

# Ruta del JDK (ajustá si cambia)
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

# Fuerza a Spark a usar el mismo Python del venv
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

## Ejercicio 1

Dado el siguiente **RDD** almacenado en la variable `rdd`:

| Partition 1 | Partition 2 | Partition 3 | Partition 4 |
| :---------: | :---------: | :---------: | :---------: |
|    34  21   |    23  45   |    3  21    |    30  91   |
|    21  34   |    12  12   |    15  10   |    31  32   |
|    10  18   |    36  18   |    14  18   |    32  53   |
|    32  45   |    4  97    |    3  15    |    19  35   |

---

### Pregunta

Responda: ¿Qué imprime cada uno de los siguientes scripts (sin ejecutarlo)?

In [ ]:
spark = SparkSession.builder \
    .appName("RDD-Ejercicio01") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

data = [
    (34, 21), (21, 34), (10, 18), (32, 45),  # Partición 1
    (23, 45), (12, 12), (36, 18), (4, 97),   # Partición 2
    (3, 21), (15, 10), (14, 18), (3, 15),    # Partición 3
    (30, 91), (31, 32), (32, 53), (19, 35)   # Partición 4
]

rdd = sc.parallelize(data, 4)

### a)

```python
res = rdd.map(lambda t: (t[0] + t[1]) * 2)
print(res.first())
```

Ese código toma cada par de números del RDD y calcula la **suma de los dos, multiplicada por dos**.
Después, muestra solo el **primer resultado** de todos esos cálculos.

En otras palabras:

> “Por cada tupla `(a, b)` del conjunto de datos, hace `(a + b) * 2`,
> y luego imprime el primer valor obtenido.”

Por ejemplo, con la primera tupla `(34, 21)` → `(34 + 21) * 2 = 110`.
Por eso, el programa imprime **110**.


In [ ]:
res = rdd.map(lambda t: (t[0] + t[1]) * 2)
print("Resultado del punto (a):", res.first())

### b)

```python
res = rdd.filter(lambda t: t[0] >= t[1])
print(res.take(3))
```

Ese código revisa todas las parejas de números del conjunto y **se queda solo con aquellas en las que el primer número es mayor o igual que el segundo**.
Después, **muestra las tres primeras parejas** que cumplen esa condición.


In [ ]:
res = rdd.filter(lambda t: t[0] >= t[1])
print(res.take(3))

### c)

```python
res = rdd.map(lambda t: (t[0], t[1], t[0] / t[1]))
res = res.filter(lambda t: t[2] < 0.5)
res = res.reduce(lambda t1, t2: t1 if t1[2] < t2[2] else t2)

print(res)
```

Primero, el código **crea una nueva lista** donde cada par de números incluye también el **resultado de dividir el primero por el segundo**.
Después, **filtra** esa lista quedándose solo con los pares donde el resultado de la división es **menor que 0.5**.
Por último, **compara todos esos casos** y se queda con el que tenga **la división más chica de todos**.

En este conjunto de datos, esa tupla resulta ser **(4, 97, 0.041)**.


In [ ]:
res = rdd.map(lambda t: (t[0], t[1], t[0] / t[1]))
res = res.filter(lambda t: t[2] < 0.5)
res = res.reduce(lambda t1, t2: t1 if t1[2] < t2[2] else t2)
print(res)

### d)

```python
r1 = rdd.map(lambda t: t[0])
r2 = rdd.map(lambda t: t[1])
r1 = r1.distinct()
r2 = r2.distinct()
res = r2.union(r1)
print(res.collect())
```

Ese código **separa** todos los primeros números del conjunto en una lista y todos los segundos números en otra.
Luego, en cada lista **elimina los valores repetidos** para quedarse solo con los distintos.
Después **une ambas listas** en una sola (sin eliminar duplicados entre ellas)
y finalmente **muestra todos los valores** de esa unión.


In [ ]:
r1 = rdd.map(lambda t: t[0])
r2 = rdd.map(lambda t: t[1])
r1 = r1.distinct()
r2 = r2.distinct()
res = r2.union(r1)
print(res.collect())

---

## 2)

**Dados los siguientes scripts en Spark, dibuje el DAG correspondiente.**
**¿Qué se termina ejecutando?**

---

### a.

```python
A = sc.textFile("Caso A")
B = A.map(fMap1)
C = B.filter(fFilter1)
final = B.reduce(fReduce1)
```

textFile → map → reduce


---

### b.

```python
A = sc.textFile("Caso B")
B = A.map(fMap1)
C = B.filter(fFilter1)
B = B.map(fMap2)
D = C.filter(fFilter2)
final = D.reduce(fReduce1)
```

textFile → map(fMap1) → filter(fFilter1) → filter(fFilter2) → reduce(fReduce1)


---

### c.

```python
A = sc.textFile("Caso C")
B = A.map(fMap1)
C = B.filter(fFilter1)
D = B.filter(fFilter1)
E = B.filter(fFilter1)
C = C.union(D)
D = C.intersection(E)
E = C.subtract(E)
final = E.reduce(fReduce1)
```

```
textFile("Caso C")
        ↓
      map(fMap1)
        ↓
 ┌────────┬────────┬────────┐
 │        │        │        │
filter    filter   filter   (tres ramas distintas sobre B)
 (C)       (D)      (E)
 │          │        │
 └──────┐   │        │
        │   │        │
      union(D)       │
        │            │
   intersection(E)    │
        │             │
      subtract(E) <───┘
        ↓
     reduce(fReduce1)

```

---

### d.

```python
A = sc.textFile("Caso D.1")
B = sc.textFile("Caso D.2")
C = sc.textFile("Caso D.3")
A = A.map(fMap1)
B = B.filter(fFilter1)
D = A.filter(fFilter1)
E = D.map(fFilter1)
F = D.union(A).union(E)
final = D.count()
```

```
[ sc.textFile("Caso D.1") ]
           ↓
       [ map(fMap1) ]
           ↓
     [ filter(fFilter1) ]
           ↓
          [ count ]
```


---

## Ejercicio 3)

Usando el dataset **Banco**, escriba un script en Python usando **Spark** para responder a las siguientes preguntas:

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Banco Spark - Local") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")
inputDir = r"C:\Users\Fabian\Desktop\big-data\Datasets_spark\\"
clientes_file = inputDir + "Clientes.txt"
cajas_file = inputDir + "CajasDeAhorro.txt"
prestamos_file = inputDir + "Prestamos.txt"
movimientos_file = inputDir + "Movimientos.txt"

def lineToTuple(linea):
    return linea.strip().split('\t')

clientes = sc.textFile(clientes_file).map(lineToTuple)
cajas = sc.textFile(cajas_file).map(lineToTuple)
prestamos = sc.textFile(prestamos_file).map(lineToTuple)
movimientos = sc.textFile(movimientos_file).map(lineToTuple)

# Cliente: <ID_Cliente, nombre, apellido, DNI, fecha de nacimiento, nacionalidad>
clientes = clientes.map(lambda t: (t[1] + " " + t[2], t[4].split("-"), t[5]))
clientes = clientes.map(lambda t: (t[0], int(t[1][0]), int(t[1][1]), int(t[1][2]), t[2]))
# CajaDeAhorro: <ID_Caja, ID_Cliente, saldo>
cajas = cajas.map(lambda c: (c[0], c[1], float(c[2])))
# Prestamos: <ID_Caja, cuotas, monto>
prestamos = prestamos.map(lambda t: (t[0], int(t[1]), float(t[2])))
# Movimientos: <ID_Caja, monto, timestamp>
movimientos = movimientos.map(lambda m: (m[0], float(m[1]), m[2]))

---
### a. Nombre y apellidos de los clientes capricornianos.

Clientes capricornianos (22/12 al 19/1)

In [ ]:
capricornianos = clientes.filter(lambda t: (t[2] == 1 and t[3] < 20) or (t[2] == 12 and t[3] > 21)) \
                         .map(lambda t: t[0])
print("a) Clientes capricornianos:")
for nombre in capricornianos.take(10):
    print(nombre)

### b. Nombre y apellido de los clientes de nacionalidad argentina.

In [ ]:
argentinos = clientes.filter(lambda t: t[4] == 'ARG')
nombres_argentinos = argentinos.map(lambda t: t[0])

print("b) Clientes argentinos:")

total_arg = nombres_argentinos.count()
print(f"Total de clientes argentinos: {total_arg}")
print("Ejemplos:")
for nombre in nombres_argentinos.take(10):
    print(" -", nombre)


### c. Del resultado de a), ¿cuántos nacieron en verano?

In [ ]:
en_verano = capricornianos.filter(lambda t: t is not None)
print("c) Cantidad de capricornianos nacidos en verano:", en_verano.count())

### d. Del resultado de b), ¿quién es el cliente más joven y quién el más viejo?

In [ ]:
mas_joven = argentinos.sortBy(lambda t: (t[1], t[2], t[3]), ascending=False).first()
mas_viejo = argentinos.sortBy(lambda t: (t[1], t[2], t[3]), ascending=True).first()
print("d) Cliente más joven:", mas_joven[0])
print("   Cliente más viejo:", mas_viejo[0])

### e. El ID de la caja que tiene asociado el préstamo con mayor cantidad de cuotas y, entre las que tienen la misma cantidad, el de mayor monto.

In [ ]:
max_cuotas = prestamos.reduce(lambda p1, p2: p1 if p1[1] > p2[1] else p2)
mas_cuotas = prestamos.filter(lambda p: p[1] == max_cuotas[1])
mayor_monto = mas_cuotas.reduce(lambda p1, p2: p1 if p1[2] > p2[2] else p2)
print("e) Caja con préstamo más largo y de mayor monto:", mayor_monto)

### f. Los ID de clientes (únicos) con al menos una caja de ahorro (en positivo) cuyo saldo es mayor a 300 U$$.

In [ ]:
clientes_300 = cajas.filter(lambda c: c[2] > 300).map(lambda c: c[1]).distinct()

print("f) IDs de clientes con saldo > 300:")

total_clientes_300 = clientes_300.count()
print(f"Total de clientes con saldo > 300: {total_clientes_300}")
print("Ejemplos:")
for cid in clientes_300.take(10):
    print(" -", cid)


### g. Del dataset **Movimientos**, el monto del mayor movimiento y el ID de caja del último movimiento.

In [ ]:
mayor_mov = movimientos.reduce(lambda m1, m2: m1 if m1[1] > m2[1] else m2)
ultimo_mov = movimientos.reduce(lambda m1, m2: m1 if m1[2] > m2[2] else m2)

print("g) Monto del mayor movimiento:", mayor_mov[1])
print("   ID de caja del último movimiento:", ultimo_mov[0])


---

## 4)

Es posible resolver los siguientes problemas (por separado) utilizando una única función **reduce**:

- a. El promedio de edades de los clientes.
- b. Determinar la cantidad de cuentas con saldo positivo y la cantidad de cuentas con saldo negativo.


Sí, ambos problemas pueden resolverse utilizando una **única función `reduce`**, ya que esta permite **recorrer todo el RDD acumulando resultados parciales** en una sola pasada.

* **a)** En el caso del promedio de edades, `reduce` puede acumular simultáneamente la **suma total de las edades** y la **cantidad de clientes**, devolviendo una tupla `(suma, cantidad)`.
  Al finalizar, el promedio se obtiene dividiendo `suma / cantidad`.
  De este modo, no es necesario aplicar transformaciones adicionales como `count()` o `sum()`.

* **b)** Para contar cuentas con saldo positivo y negativo, cada elemento del RDD puede representarse como una tupla `(positivos, negativos)`, donde se suma `(1, 0)` si el saldo es positivo y `(0, 1)` si es negativo.
  La función `reduce` combina todos los pares sumando los valores de cada posición, resultando en `(total_positivos, total_negativos)`.

En ambos casos, la función `reduce` permite **resolver el problema en una sola operación de agregación**, evitando múltiples pasadas sobre los datos y aprovechando el **procesamiento distribuido** de Spark.

---

## 5)

El dataset **EstacionesMeteorológicas** posee información sobre registros de datos climáticos tomados por sus estaciones.
Este dataset tiene tuplas con la siguiente información:

```
<ID_Estación, fecha_registro, temperatura, humedad, precipitación>
```

Y además está conformado por dos archivos:

**a.** `estacionNorte.txt` almacena la información en grados centígrados, porcentaje de humedad y milímetros de lluvia.
**b.** `estacionSur.txt` almacena la información en grados Fahrenheit, porcentaje de humedad y centímetros de lluvia.

---

Implemente una solución en **Spark** que permita obtener:

* El **promedio** de temperatura, de humedad y de precipitación total entre todas las estaciones.
* El **ID de la estación** y la **fecha** que registró:

  * la temperatura más fría
  * la temperatura más calurosa
  * la de mayor humedad
  * la de menor humedad
  * la de más precipitación
  * la de menor precipitación

---

**NOTA:**
En caso de dos estaciones con igual máximo o mínimo, devolver cualquiera de las dos.

---


In [ ]:
from pyspark.sql import SparkSession
import os, sys

# ===============================
# CONFIGURACIÓN DE ENTORNO (Windows + Spark)
# ===============================
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# ===============================
# INICIO DE SPARK
# ===============================
spark = SparkSession.builder \
    .appName("EstacionesMeteorologicasLocal") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

# ===============================
# RUTAS LOCALES
# ===============================
inputDir = r"C:\Users\Fabian\Desktop\big-data\Datasets_spark\EstacionesMeteorologicas\\"

# ===============================
# FUNCIONES AUXILIARES
# ===============================
def splitter(line):
    """Divide una línea por tabulación"""
    return line.strip().split('\t')

def to_celsius(fahrenheit):
    """Convierte Fahrenheit a Celsius"""
    return round((float(fahrenheit) - 32) / 1.8, 2)

def to_mm(cm):
    """Convierte cm a mm"""
    return round(float(cm) * 10, 2)

# ===============================
# CARGA DE ARCHIVOS
# ===============================
print("Cargando archivos desde carpetas locales...")

import glob

def load_folder(path):
    files = glob.glob(os.path.join(path, "*.txt"))
    rdds = [sc.textFile(f).map(splitter) for f in files]
    return sc.union(rdds)

norte = load_folder(os.path.join(inputDir, "Norte"))
sur = load_folder(os.path.join(inputDir, "Sur"))

print("Archivos cargados correctamente ✅")
print(f"Total Norte: {norte.count()} | Total Sur: {sur.count()}")

# ===============================
# CONVERSIÓN DE DATOS Y UNIÓN
# ===============================
# <ID_Estación, fecha_registro, temperatura, humedad, precipitación>
# Norte → grados °C, mm
# Sur   → Fahrenheit → °C, cm → mm

norte = norte.map(lambda e: (e[0], e[1], float(e[2]), float(e[3]), float(e[4])))
sur = sur.map(lambda e: (e[0], e[1], to_celsius(e[2]), float(e[3]), to_mm(e[4])))

# Unión
estaciones = norte.union(sur).cache()
print(f"Total registros combinados: {estaciones.count()}")

# ===============================
# PARTE 1: Promedios
# ===============================
part1 = estaciones.map(lambda e: (e[2], e[3], e[4], 1))
sumas = part1.reduce(lambda a, b: (a[0]+b[0], a[1]+b[1], a[2]+b[2], a[3]+b[3]))

prom_temp = round(sumas[0] / sumas[3], 2)
prom_hum = round(sumas[1] / sumas[3], 2)
prom_prec = round(sumas[2] / sumas[3], 2)

print("\n=== PROMEDIOS ===")
print(f"Temperatura promedio: {prom_temp} °C")
print(f"Humedad promedio: {prom_hum} %")
print(f"Precipitación promedio: {prom_prec} mm")

# ===============================
# PARTE 2: Mínimos y Máximos
# ===============================
# (temp_min, temp_max, hum_max, hum_min, prec_max, prec_min)
part2 = estaciones.map(lambda e: (e, e, e, e, e, e))
part2 = part2.reduce(lambda e1, e2: (
    e1[0] if e1[0][2] < e2[0][2] else e2[0],  # temp_min
    e1[1] if e1[1][2] > e2[1][2] else e2[1],  # temp_max
    e1[2] if e1[2][3] > e2[2][3] else e2[2],  # hum_max
    e1[3] if e1[3][3] < e2[3][3] else e2[3],  # hum_min
    e1[4] if e1[4][4] > e2[4][4] else e2[4],  # prec_max
    e1[5] if e1[5][4] < e2[5][4] else e2[5]   # prec_min
))

print("\n=== EXTREMOS ===")
print(f"Temperatura más fría: {part2[0][2]} °C | Estación {part2[0][0]} | Fecha {part2[0][1]}")
print(f"Temperatura más calurosa: {part2[1][2]} °C | Estación {part2[1][0]} | Fecha {part2[1][1]}")
print(f"Mayor humedad: {part2[2][3]} % | Estación {part2[2][0]} | Fecha {part2[2][1]}")
print(f"Menor humedad: {part2[3][3]} % | Estación {part2[3][0]} | Fecha {part2[3][1]}")
print(f"Mayor precipitación: {part2[4][4]} mm | Estación {part2[4][0]} | Fecha {part2[4][1]}")
print(f"Menor precipitación: {part2[5][4]} mm | Estación {part2[5][0]} | Fecha {part2[5][1]}")

# ===============================
# FINALIZACIÓN
# ===============================
spark.stop()
print("\nEjecución completada ✅")
